# Manual case inspection: view the actual images for wrong has_error=1 grading cases

Follow-on to the raw-text inspection done manually on 2026-08-05 (see
`pilot/06_7b_error_stratum_power.ipynb`'s merge cell docstring). Reading the
raw grading text explained *how* the model got most of these wrong (digit/
reasoning self-contradiction, pattern-matching a solution's structure instead
of independently recomputing an arithmetic step) -- but two of the checked
cases (reference-run items 40 and 161) had no findable error anywhere in the
text. Worth checking those against the actual handwritten image before
concluding the ground truth is right and the model is simply wrong.

No GPU and no model needed here -- this notebook only reads the FERMAT
dataset and the existing 05/06 checkpoints, both already on Drive. It
recomputes the wrong-has_error=1 case list directly from the checkpoints (not
a hardcoded index list), so re-running it after notebook 06 finishes picks up
the new items automatically -- it works fine on a partial run too, it just
inspects however many extra items are done so far.

For every flagged case it:
1. Saves the image to `Drive/uncertainty-math-vlm/check_images/`.
2. Displays it inline.
3. Prints the question, the perturbed worked solution shown to the model,
   the model's actual reasoning text, and whether all K=5 samples gave the
   identical wrong digit (`unanimous=True`, i.e. `reasoning_entropy == 0` --
   a case entropy has no disagreement to detect even though the model is
   confidently wrong).

In [ ]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and existing checkpoints, it never runs generation.
import json
import os

from huggingface_hub import login
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"

# Reuses the token already cached on Drive by 05/06 -- will not prompt again.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import sys
sys.path.insert(0, os.path.abspath("repo"))
import importlib
importlib.invalidate_caches()

import pilot.data
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.data.__file__)}")

In [ ]:
# Reference run: rebuild the exact n=300 sample the 05/06 runs used, and load
# its raw checkpoint. Precision-searched the same way notebook 06's merge
# cell does -- guessing the filename from this session's own state (there is
# none here, this notebook loads no model) would risk a confusing "not
# found" instead of a clear error if the reference run's precision differs.
import glob

SEED = 42
MODEL_SLUG = "Qwen2.5-VL-7B-Instruct"
K_GRADING = 5

reference_sample = pilot.data.load_fermat_balanced(n=300, seed=SEED, target_error_frac=0.5)

_ref_base = f"{CHECKPOINT_DIR}/grading_7b_k{K_GRADING}_{MODEL_SLUG}_n300_seed{SEED}"
_ref_candidates = {"bf16": f"{_ref_base}.jsonl", "4bit": f"{_ref_base}_4bit.jsonl"}
_found = {label: path for label, path in _ref_candidates.items() if os.path.exists(path)}
if not _found:
    raise AssertionError(
        f"No reference checkpoint at {_ref_candidates['bf16']} or "
        f"{_ref_candidates['4bit']} -- it is the raw 2026-08-05 300-item run; "
        "without it there is nothing to inspect."
    )
REFERENCE_CHECKPOINT = next(iter(_found.values()))

with open(REFERENCE_CHECKPOINT) as f:
    reference_entries = [json.loads(line) for line in f if line.strip()]
assert len(reference_entries) == len(reference_sample), (
    f"{len(reference_entries)} checkpoint entries vs {len(reference_sample)} sample "
    "items -- checkpoint order no longer matches the sample; do not trust the "
    "index-based image lookups below until this is resolved."
)
print(f"reference: {len(reference_sample)} items, checkpoint loaded from "
      f"{os.path.basename(REFERENCE_CHECKPOINT)}")

In [ ]:
# Extra items (notebook 06): only present once that run has checkpointed at
# least one item. Fine to run this on a partial run -- it just inspects
# however many are done so far, and a re-run later picks up the rest.
SKIP = 150
extra_pattern = (f"{CHECKPOINT_DIR}/grading_7b_extra_error_k{K_GRADING}_{MODEL_SLUG}"
                  f"_n*_skip{SKIP}_seed{SEED}*.jsonl")
extra_matches = sorted(glob.glob(extra_pattern), key=os.path.getmtime, reverse=True)

extra_entries, extra_sample = [], None
if extra_matches:
    EXTRA_CHECKPOINT = extra_matches[0]
    with open(EXTRA_CHECKPOINT) as f:
        raw = [json.loads(line) for line in f if line.strip()]
    # Only fully-generated entries -- an interrupted write could leave a
    # partial last line, and a partial entry would misalign the index-based
    # sample lookup below (extra_sample[idx] assumes 1:1 order with the file).
    extra_entries = [e for e in raw if len(e.get("samples_raw", [])) == K_GRADING]
    if extra_entries:
        extra_sample = pilot.data.load_fermat_extra_error_items(
            n_extra=len(extra_entries), seed=SEED, skip=SKIP
        )
    print(f"extra: {os.path.basename(EXTRA_CHECKPOINT)} -- "
          f"{len(extra_entries)} complete items so far")
else:
    print("No extra-items checkpoint found yet -- inspecting the reference run only.")

In [ ]:
# Score every entry the same way notebook 06's merge cell does (duplicated
# deliberately -- each notebook here is self-contained by project
# convention), and flag every case where has_error=1 but the model's
# majority vote said 0.
def score_entry(entry):
    digits = [pilot.parsing.parse_grading(t) for t in entry["samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, count = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    return {
        "orig_q": entry["item"]["orig_q"],
        "pert_a": entry["item"]["pert_a"],
        "has_error": bool(entry["item"]["has_error"]),
        "said_error": said_error,
        "grading_correct": majority in {"0", "1"} and said_error == bool(entry["item"]["has_error"]),
        "unanimous": count == len(labels),   # True <=> reasoning_entropy == 0 for this item
        "parsed_digits": digits,
        "samples_raw": entry["samples_raw"],
    }


def flag_wrong_error_cases(entries, sample, source_label):
    flagged = []
    for idx, entry in enumerate(entries):
        s = score_entry(entry)
        if s["has_error"] and not s["grading_correct"]:
            flagged.append({"source": source_label, "idx": idx, "sample": sample, **s})
    return flagged


flagged = flag_wrong_error_cases(reference_entries, reference_sample, "reference")
if extra_sample is not None:
    flagged += flag_wrong_error_cases(extra_entries, extra_sample, "extra")

n_unanimous = sum(f["unanimous"] for f in flagged)
print(f"{len(flagged)} wrong has_error=1 cases total "
      f"({n_unanimous} unanimous across all {K_GRADING} samples, i.e. entropy == 0)")
for f in flagged:
    tag = "UNANIMOUS" if f["unanimous"] else "split vote"
    print(f"  [{f['source']:9s} idx {f['idx']:3d}]  {tag:10s}  {f['orig_q'][:65].strip()!r}")

In [ ]:
# Save every flagged case's actual handwritten image to Drive, and display
# it inline next to the question / worked solution / model's own reasoning
# text -- so the text-based reading (self-contradiction, pattern-matching
# instead of recomputing, or -- for #40 and #161 in the reference run -- no
# findable error at all) can be checked against what was actually shown.
import matplotlib.pyplot as plt

OUT_DIR = f"{PROJECT_DIR}/check_images"
os.makedirs(OUT_DIR, exist_ok=True)

for f in flagged:
    item = f["sample"][f["idx"]]
    img = item["image"]
    fname = f"{f['source']}_idx{f['idx']}_unanimous{f['unanimous']}.png"
    img.save(f"{OUT_DIR}/{fname}")

    print("=" * 78)
    print(f"[{f['source']} idx {f['idx']}]  has_error=1, model majority said "
          f"{'error' if f['said_error'] else 'no error'}  unanimous={f['unanimous']}")
    print(f"Q: {f['orig_q'].strip()}")
    print(f"pert_a shown to model: {f['pert_a'].strip()[:400]}")
    reasoning = pilot.parsing.parse_grading_reasoning(f["samples_raw"][0])
    print(f"model's reasoning (sample 0 of {K_GRADING}): {reasoning}")
    print(f"saved -> check_images/{fname}")

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{f['source']} idx {f['idx']}  unanimous={f['unanimous']}")
    plt.show()

print(f"\nAll {len(flagged)} images saved to Drive: "
      "My Drive > uncertainty-math-vlm > check_images")